# 예제 03. 사용자 정의 Dataset
빅데이터프로그래밍 · 5주차

## 목표
- `__init__` `__len__` `__getitem__` 의 역할을 이해한다
- Dataset 클래스를 직접 작성한다
- CSV에서 읽어오는 Dataset을 만든다

세 메서드만 만들면 PyTorch가 나머지를 알아서 처리합니다.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


## 1. 가장 단순한 형태

| 메서드 | 역할 | 언제 불리는가 |
| --- | --- | --- |
| `__init__` | 데이터를 준비한다 | 객체를 만들 때 한 번 |
| `__len__` | 전체 개수를 알려준다 | `len(dataset)` |
| `__getitem__` | i번째 데이터 하나를 돌려준다 | `dataset[i]` · DataLoader가 반복 호출 |


In [ ]:
class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


X = torch.randn(12, 4)
y = torch.randint(0, 2, (12, 1)).float()

ds = MyDataset(X, y)
print("개수:", len(ds))
print("3번째:", ds[3][0].shape, ds[3][1])


In [ ]:
loader = DataLoader(ds, batch_size=4, shuffle=True)

for bx, by in loader:
    print(tuple(bx.shape), tuple(by.shape))


## 2. __getitem__ 이 언제 불리는지 눈으로 보기
DataLoader는 필요할 때마다 하나씩 꺼냅니다. 전체를 메모리에 올리지 않아도 되는 이유입니다.


In [ ]:
class LoudDataset(Dataset):
    def __init__(self, n):
        self.n = n

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        print(f"  __getitem__({idx}) 호출")
        return torch.tensor([float(idx)])


loud = LoudDataset(6)
print("DataLoader 생성 — 아직 아무것도 안 불립니다")
loud_loader = DataLoader(loud, batch_size=3, shuffle=False)

print("\n반복 시작")
for (b,) in loud_loader:
    print("batch:", b.flatten().tolist())


## 3. CSV에서 읽어오는 Dataset
3주차 전처리를 `__init__` 안에 넣습니다.


In [ ]:
csv = """study_hours,attendance,midterm,pass
12.5,95,88,1
8.0,88,92,1
4.5,61,61,0
15.0,100,95,1
6.5,74,70,0
10.0,90,84,1
9.5,85,66,0
13.0,97,100,1
"""
with open("scores.csv", "w") as f:
    f.write(csv)
print("scores.csv 생성")


In [ ]:
import pandas as pd

class CSVDataset(Dataset):
    def __init__(self, path, target="pass"):
        df = pd.read_csv(path)
        self.columns = [c for c in df.columns if c != target]

        X = df[self.columns].to_numpy(dtype="float32")
        y = df[[target]].to_numpy(dtype="float32")

        # 표준화 — 전처리는 __init__ 에서 한 번만
        self.mean, self.std = X.mean(axis=0), X.std(axis=0)
        X = (X - self.mean) / self.std

        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


ds = CSVDataset("scores.csv")
print("개수:", len(ds), "/ 입력 열:", ds.columns)
print("한 개:", ds[0])


In [ ]:
loader = DataLoader(ds, batch_size=3, shuffle=True)
for bx, by in loader:
    print(tuple(bx.shape), tuple(by.shape))


## 4. 학습용과 검증용으로 나누기
`random_split` 이 Dataset을 그대로 두고 인덱스만 나눠 줍니다.


In [ ]:
from torch.utils.data import random_split

n_train = int(len(ds) * 0.75)
n_val = len(ds) - n_train

train_ds, val_ds = random_split(ds, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=2, shuffle=False)

print("학습:", len(train_ds), "검증:", len(val_ds))
print("학습 batch 수:", len(train_loader))


## 직접 해보기
1. `__getitem__` 이 입력만 돌려주도록 바꾸면 DataLoader 반복문은 어떻게 달라지나요?
2. `CSVDataset` 에 `attendance` 열을 빼는 옵션을 추가하세요.


In [ ]:
# 여기에 작성하세요
